# Abalone rings prediction and experiment tracking

Predict `Rings` from the eight original predictors. Following PR 1, start with a simple working regression model and record reproducible experiments with MLflow. This notebook follows the course's `DictVectorizer`, sklearn pipeline, and explicit MLflow logging pattern.

The raw observations, including the four height values flagged in EDA, are retained for this first baseline. No unverified corrections or target-based filtering are applied. More elaborate cleaning and tree-based models are optional follow-up experiments, not prerequisites for this first model.

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
ROOT = Path.cwd().resolve()
if not (ROOT / "data/abalone.csv").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/abalone.csv"
assert DATA_PATH.exists(), "Run from the repository root or notebooks directory."
WORK_PATH = ROOT / "work"
WORK_PATH.mkdir(exist_ok=True)

## Load data and explicitly separate features from the target

Use an explicit feature list so a derived column such as `Age = Rings + 1.5` can never enter the inputs accidentally. `Sex` is categorical; the other seven inputs are measurements. Predictions remain continuous estimates of rings, without rounding before evaluation.

In [ ]:
FEATURES = ["Sex", "Length", "Diameter", "Height", "Whole weight", "Shucked weight", "Viscera weight", "Shell weight"]
TARGET = "Rings"
df = pd.read_csv(DATA_PATH)
assert df.columns.tolist() == FEATURES + [TARGET], "Unexpected dataset schema"
assert not df[FEATURES + [TARGET]].isna().any().any(), "Review missing values before fitting."
assert df["Sex"].isin(["F", "I", "M"]).all(), "Unexpected Sex category"
assert "Age" not in FEATURES and TARGET not in FEATURES
X = df[FEATURES].copy()
y = df[TARGET].copy()
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print(f"Rows: {len(df):,}; predictors: {len(FEATURES)}; dataset SHA256: {DATA_SHA256}")
X.head()

## Split before learning any preprocessing

A fixed random split allocates approximately 60% to training, 20% to validation, and 20% to final testing. All candidates use exactly the same rows. Fit encoders and scaling on training data only. Select by validation RMSE; the final test is reserved for the selected model.

EDA already examined the full dataset, so this is a held-out model evaluation rather than a completely untouched external validation dataset. It estimates performance on similar observations; it does not establish performance on a new population.

In [ ]:
X_development, X_test, y_development, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.25, random_state=RANDOM_STATE
)
assert set(X_train.index).isdisjoint(X_validation.index)
assert set(X_development.index).isdisjoint(X_test.index)
train_records = X_train.to_dict(orient="records")
validation_records = X_validation.to_dict(orient="records")
print({"train": len(X_train), "validation": len(X_validation), "test": len(X_test)})

## Define small, interpretable experiments

`DictVectorizer` converts records to numeric features and one-hot encodes `Sex`, as in the course taxi exercise. A pipeline retains the fitted encoder together with its regressor for later prediction.

- **Mean baseline:** always predicts the training target mean. A useful model should improve on this.
- **Linear regression:** the simple model explicitly allowed by the assignment.
- **Ridge:** a small regularized alternative motivated by correlated measurements in EDA. Use a fixed `alpha=1.0`, not a parameter search. Standardize numeric inputs inside this pipeline because the penalty depends on feature scale. Scaling does not remove skewness.

The unregularized linear fit need not have uniquely interpretable coefficients when encoded categories and correlated measurements overlap. The goal here is prediction, not causal interpretation.

In [ ]:
candidates = {
    "mean_baseline": Pipeline(
        [
            ("encoder", DictVectorizer(sparse=False)),
            ("regressor", DummyRegressor(strategy="mean")),
        ]
    ),
    "linear_regression": Pipeline(
        [
            ("encoder", DictVectorizer(sparse=False)),
            ("regressor", LinearRegression()),
        ]
    ),
    "ridge_alpha_1": Pipeline(
        [
            ("encoder", DictVectorizer(sparse=False)),
            ("scaler", StandardScaler()),
            ("regressor", Ridge(alpha=1.0)),
        ]
    ),
}


def regression_metrics(y_true: pd.Series, prediction: np.ndarray) -> dict[str, float]:
    """Measure ring prediction errors and explained variance."""
    return {
        "mae": float(mean_absolute_error(y_true, prediction)),
        "rmse": float(root_mean_squared_error(y_true, prediction)),
        "r2": float(r2_score(y_true, prediction)),
    }

## Track each candidate with MLflow

Store runs under the repository's ignored `mlruns/` directory. Each run records the dataset hash, feature list, split seed, estimator parameters, train/validation metrics, and fitted pipeline. No tracking files or fitted models are committed.

MAE is average absolute error in rings. RMSE penalizes large errors more strongly; it does not directly give greater weight to older animals. R² compares squared error against a constant reference and may be negative.

In [ ]:
mlflow.set_tracking_uri((ROOT / "mlruns").as_uri())
mlflow.set_experiment("abalone-rings-notebook")
results = []
for name, pipeline in candidates.items():
    with mlflow.start_run(run_name=name) as run:
        mlflow.set_tags({"stage": "validation", "dataset_sha256": DATA_SHA256, "target": TARGET})
        mlflow.log_params(
            {
                "model": name,
                "random_state": RANDOM_STATE,
                "test_fraction": 0.20,
                "validation_fraction": 0.20,
                "height_policy": "retain_original_observations",
                "features": json.dumps(FEATURES),
                "train_rows": len(X_train),
                "validation_rows": len(X_validation),
            }
        )
        mlflow.log_params({f"regressor_{key}": value for key, value in pipeline[-1].get_params().items()})
        pipeline.fit(train_records, y_train)
        train_metrics = regression_metrics(y_train, pipeline.predict(train_records))
        validation_metrics = regression_metrics(y_validation, pipeline.predict(validation_records))
        mlflow.log_metrics({f"train_{key}": value for key, value in train_metrics.items()})
        mlflow.log_metrics({f"validation_{key}": value for key, value in validation_metrics.items()})
        mlflow.sklearn.log_model(pipeline, artifact_path="model")
        results.append(
            {
                "model": name,
                "run_id": run.info.run_id,
                **{f"validation_{key}": value for key, value in validation_metrics.items()},
            }
        )

comparison = pd.DataFrame(results).sort_values("validation_rmse").reset_index(drop=True)
comparison

## Select on validation, refit, and evaluate once on test

Choose the smallest validation RMSE, then refit that pipeline on training plus validation rows. Evaluate this final fit on the held-out test rows once. Do not change models or tune settings using this test result. Record the final model separately so it is not confused with the smaller training-only fits.

In [ ]:
selected_name = str(comparison.loc[0, "model"])
final_pipeline = clone(candidates[selected_name])
development_records = X_development.to_dict(orient="records")
test_records = X_test.to_dict(orient="records")
with mlflow.start_run(run_name=f"final_{selected_name}") as final_run:
    mlflow.set_tags({"stage": "final_test", "dataset_sha256": DATA_SHA256, "target": TARGET})
    mlflow.log_params(
        {
            "model": selected_name,
            "selection_metric": "validation_rmse",
            "selection_run_id": str(comparison.loc[0, "run_id"]),
            "random_state": RANDOM_STATE,
            "development_rows": len(X_development),
            "test_rows": len(X_test),
            "features": json.dumps(FEATURES),
            "height_policy": "retain_original_observations",
        }
    )
    final_pipeline.fit(development_records, y_development)
    test_predictions = final_pipeline.predict(test_records)
    test_metrics = regression_metrics(y_test, test_predictions)
    mlflow.log_metrics({f"test_{key}": value for key, value in test_metrics.items()})
    mlflow.sklearn.log_model(final_pipeline, artifact_path="model")
    final_run_id = final_run.info.run_id
    final_model_uri = f"runs:/{final_run_id}/model"

print(f"Selected model: {selected_name}")
pd.Series(test_metrics, name="held_out_test")

## Inspect errors without using them for model selection

Inspect prediction spread and residuals. Positive residuals mean underprediction. Group summaries are descriptive and can be unreliable for small groups. They are not new model-selection criteria.

In [ ]:
errors = X_test.copy()
errors["actual_rings"] = y_test
errors["predicted_rings"] = test_predictions
errors["residual"] = errors["actual_rings"] - errors["predicted_rings"]
errors["absolute_error"] = errors["residual"].abs()
errors["rings_group"] = pd.cut(errors["actual_rings"], bins=[0, 7, 11, 15, np.inf], labels=["1-7", "8-11", "12-15", "16+"])
display(
    errors.groupby("rings_group", observed=True).agg(
        observations=("actual_rings", "size"), mae=("absolute_error", "mean"), mean_residual=("residual", "mean")
    )
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test, test_predictions, alpha=0.3, s=12)
limits = [min(y_test.min(), test_predictions.min()), max(y_test.max(), test_predictions.max())]
axes[0].plot(limits, limits, linestyle="--", color="black")
axes[0].set(xlabel="Actual rings", ylabel="Predicted rings", title="Held-out predictions")
axes[1].scatter(test_predictions, errors["residual"], alpha=0.3, s=12)
axes[1].axhline(0, linestyle="--", color="black")
axes[1].set(xlabel="Predicted rings", ylabel="Actual minus predicted rings", title="Held-out residuals")
plt.tight_layout()
plt.show()

## Reload the saved model and reproduce predictions

Use MLflow's sklearn flavor to load the complete fitted pipeline, then pass records containing exactly the original predictors. The check demonstrates that the stored encoder and model can be reused together. Pickle export for the web service belongs to PR 2, not this notebook stage.

In [ ]:
loaded_pipeline = mlflow.sklearn.load_model(final_model_uri)
reloaded_predictions = loaded_pipeline.predict(test_records)
np.testing.assert_allclose(reloaded_predictions, test_predictions)
report = {
    "selected_model": selected_name,
    "dataset_sha256": DATA_SHA256,
    "split_rows": {"train": len(X_train), "validation": len(X_validation), "test": len(X_test)},
    "comparison": comparison.to_dict(orient="records"),
    "test_metrics": test_metrics,
    "final_run_id": final_run_id,
    "model_uri": final_model_uri,
    "reload_predictions_match": True,
    "error_by_rings_group": errors.groupby("rings_group", observed=True)
    .agg(observations=("actual_rings", "size"), mae=("absolute_error", "mean"), mean_residual=("residual", "mean"))
    .reset_index()
    .to_dict(orient="records"),
}
(WORK_PATH / "modelling_results.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved model reload reproduces held-out predictions.")
print(f"Model URI: {final_model_uri}")
print(f"Open tracking UI from repository root: uv run mlflow ui --backend-store-uri {ROOT / 'mlruns'} --port 5002")

## Conclusions and handoff to PR 2

- The table above compares a constant baseline, ordinary linear regression, and a fixed Ridge alternative on the same validation split. The selected name and final held-out scores are printed above and saved locally in `work/modelling_results.json`.
- Report MAE and RMSE in rings, together with R². The group error table reveals whether the average score hides larger errors among rare high-ring observations.
- All eight predictors are explicit. `Age` and `Rings` are excluded from model inputs. Encoder and optional scaling are fitted inside each pipeline, avoiding preprocessing leakage.
- No records were removed based on height or the target. Measurement anomalies, correlated predictors, and generalization outside this dataset remain limitations. Test diagnostics must not be used to repeatedly optimize the same held-out split.
- For PR 2, move the selected training recipe and its record-based prediction interface into the provided Python modules, then add the assignment's pickle export. Prefect scheduling and FastAPI/Docker implementation belong to later stages.
- To reproduce: run this notebook top to bottom in the uv environment. MLflow logs stay local and ignored; repeated execution creates new run IDs.

## Observed results for the bundled dataset

With seed 42, the split contains 2,505 training, 836 validation, and 836 test rows. Validation RMSE was 3.1686 for the mean baseline, 2.4198 for linear regression, and 2.4210 for Ridge. Linear regression was selected by the predefined criterion; its very small margin over Ridge is not evidence of a statistically significant difference.

After refitting on the 3,341 development rows, the selected linear model obtained test MAE **1.5931 rings**, RMSE **2.2116 rings**, and R² **0.5482**. The model reload check reproduced predictions. These figures describe this fixed split and dataset, not a guaranteed production accuracy.

Errors are uneven: the 54 test observations with at least 16 rings had MAE **4.8395**, with mean residual **+4.7820**, indicating substantial underprediction. The 473 observations with 8–11 rings had MAE **1.2497**. This supports reporting error by target range alongside aggregate metrics. No further model choices were made from these test diagnostics.
